# Healthcare Vitals and Readmission Lakehouse

This notebook loads the project's anonymized CSV outputs from a Unity Catalog
Volume, validates and models admissions data with Spark, trains a baseline
readmission classifier, and includes an optional synthetic live-vitals stream.



The streaming demo uses Spark's rate source; it is synthetic and is not a
connection to the dashboard's in-process Kafka simulator.


In [0]:
|from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

CATALOG = "workspace"
SCHEMA = "healthcare_capstone"
VOLUME = "project_data"
INPUT_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
OUTPUT_SCHEMA = f"{CATALOG}.{SCHEMA}"
CHECKPOINT_ROOT = f"{INPUT_ROOT}/checkpoints"

print(f"Input data: {INPUT_ROOT}")
print(f"Managed tables: {OUTPUT_SCHEMA}")


Input data: /Volumes/workspace/healthcare_capstone/project_data
Managed tables: workspace.healthcare_capstone


## 1. Load project data

The uploaded extracts contain pseudonymized patient hashes and masked SSNs.
Do not upload raw patient identifiers or secrets to this Volume.


In [0]:
fact_path = f"{INPUT_ROOT}/gold/fact_admissions_all.csv"
patient_path = f"{INPUT_ROOT}/gold/dim_patient.csv"
doctor_path = f"{INPUT_ROOT}/gold/dim_doctor.csv"
diagnosis_path = f"{INPUT_ROOT}/gold/dim_diagnosis.csv"

required_paths = [fact_path, patient_path, doctor_path, diagnosis_path]
required_files = {path.rsplit("/", 1)[-1] for path in required_paths}
available_files = {item.name.rstrip("/") for item in dbutils.fs.ls(f"{INPUT_ROOT}/gold")}
missing_paths = sorted(required_files.difference(available_files))
if missing_paths:
    raise FileNotFoundError(
        f"Required project CSVs are missing from {INPUT_ROOT}/gold: " + ", ".join(missing_paths)
    )

def read_project_csv(path):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )

source_fact = read_project_csv(fact_path)
dim_patient_source = read_project_csv(patient_path)
dim_doctor_source = read_project_csv(doctor_path)
dim_diagnosis_source = read_project_csv(diagnosis_path)

display(source_fact.limit(10))


admission_id,patient_hash,doctor_id,diagnosis_code,admission_date,discharge_date,admission_month,admission_type,length_of_stay,prior_inpatient_admissions,num_medications,num_lab_procedures,num_abnormal_labs,avg_a1c_result,total_charges,insurance_type,icu_stay,readmit_30d
ADM_000001,98283da5f25d4bc1,DOC_015,J18.9,2025-10-31,2025-11-02,2025-10-01,Urgent,2,3,17,38,1,78.4,8483.78,Medicaid,1,1
ADM_000002,4145d5e8212f75fa,DOC_004,K52.9,2025-03-30,2025-04-04,2025-03-01,Emergency,5,2,12,23,0,58.55,16700.16,Private Commercial,0,0
ADM_000003,53202b4e86f99985,DOC_015,K52.9,2025-08-28,2025-09-04,2025-08-01,Urgent,7,0,25,38,1,81.1,17309.79,Self-Pay,0,1
ADM_000004,3fabb75529936c8c,DOC_013,S06.0,2025-10-09,2025-10-12,2025-10-01,Emergency,3,0,25,34,1,55.95,5591.45,Private Commercial,0,0
ADM_000005,12586f15394869f9,DOC_022,J44.9,2025-09-10,2025-09-13,2025-09-01,Emergency,3,2,18,36,2,62.55,8471.48,Self-Pay,1,1
ADM_000006,5df6155c1797a810,DOC_014,S06.0,2025-09-13,2025-09-15,2025-09-01,Trauma Center,2,0,23,49,0,100.15,4676.67,Medicaid,0,0
ADM_000007,7ab239d8e7b35338,DOC_014,S06.0,2025-12-22,2025-12-24,2025-12-01,Trauma Center,2,0,7,11,2,115.75,5965.29,Self-Pay,0,0
ADM_000008,f8ec53d8f48f506c,DOC_005,J18.9,2025-03-04,2025-03-08,2025-03-01,Urgent,4,1,19,43,1,81.85,7621.65,Self-Pay,0,0
ADM_000009,d94c12df8a0b2e6b,DOC_009,M54.5,2025-02-06,2025-02-10,2025-02-01,Urgent,4,2,8,12,2,94.9,7198.21,Self-Pay,0,0
ADM_000010,5fe22b3ac56d671c,DOC_025,M54.5,2025-04-24,2025-04-27,2025-04-01,Elective,3,1,24,11,2,110.6,9699.92,Medicare,0,0


## 2. Bronze ingestion and data-quality checks


In [0]:
required_columns = {
    "admission_id",
    "patient_hash",
    "doctor_id",
    "diagnosis_code",
    "admission_month",
    "length_of_stay",
    "prior_inpatient_admissions",
    "num_medications",
    "num_lab_procedures",
    "num_abnormal_labs",
    "avg_a1c_result",
    "total_charges",
    "icu_stay",
    "readmit_30d",
}
missing_columns = required_columns.difference(source_fact.columns)
if missing_columns:
    raise ValueError(f"Admissions extract is missing required columns: {sorted(missing_columns)}")

bronze_fact = source_fact
bronze_fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{OUTPUT_SCHEMA}.capstone_bronze_admissions"
)

bronze_counts = spark.createDataFrame(
    [("admissions", bronze_fact.count())],
    ["dataset", "row_count"],
)
display(bronze_counts)


dataset,row_count
admissions,1200


## 3. Silver cleansing and enrichment

Duplicate admission IDs and rows missing model-critical values are excluded.


In [0]:
silver_fact = (
    bronze_fact
    .dropDuplicates(["admission_id"])
    .withColumn("length_of_stay", F.col("length_of_stay").cast(IntegerType()))
    .withColumn("prior_inpatient_admissions", F.col("prior_inpatient_admissions").cast(IntegerType()))
    .withColumn("num_medications", F.col("num_medications").cast(IntegerType()))
    .withColumn("num_lab_procedures", F.col("num_lab_procedures").cast(IntegerType()))
    .withColumn("num_abnormal_labs", F.col("num_abnormal_labs").cast(IntegerType()))
    .withColumn("avg_a1c_result", F.col("avg_a1c_result").cast(DoubleType()))
    .withColumn("total_charges", F.col("total_charges").cast(DoubleType()))
    .withColumn("icu_stay", F.col("icu_stay").cast(IntegerType()))
    .withColumn("readmit_30d", F.col("readmit_30d").cast(IntegerType()))
    .filter(
        F.col("admission_id").isNotNull()
        & F.col("patient_hash").isNotNull()
        & F.col("readmit_30d").isin(0, 1)
    )
)

silver_fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{OUTPUT_SCHEMA}.capstone_silver_admissions"
)

silver_patient = dim_patient_source.dropDuplicates(["patient_hash"])
silver_doctor = dim_doctor_source.dropDuplicates(["doctor_id"])
silver_diagnosis = dim_diagnosis_source.dropDuplicates(["diagnosis_code"])

print(f"Bronze rows: {bronze_fact.count():,}; silver rows: {silver_fact.count():,}")
display(silver_fact.limit(10))


Bronze rows: 1,200; silver rows: 1,200


admission_id,patient_hash,doctor_id,diagnosis_code,admission_date,discharge_date,admission_month,admission_type,length_of_stay,prior_inpatient_admissions,num_medications,num_lab_procedures,num_abnormal_labs,avg_a1c_result,total_charges,insurance_type,icu_stay,readmit_30d
ADM_000013,9fef0090d016f524,DOC_008,E11.9,2025-06-24,2025-06-30,2025-06-01,Urgent,6,0,15,22,1,66.75,19352.83,Private Commercial,0,0
ADM_000014,8be17597f368be8e,DOC_008,I21.9,2025-11-27,2025-11-30,2025-11-01,Elective,3,0,9,74,0,52.3,8102.22,Self-Pay,0,0
ADM_000076,a11888eef15ba196,DOC_012,I21.9,2025-11-02,2025-11-05,2025-11-01,Elective,3,1,27,52,2,131.9,8242.4,Private Commercial,0,0
ADM_000099,4793b95e4a1e4622,DOC_001,J18.9,2025-04-03,2025-04-08,2025-04-01,Urgent,5,0,27,43,1,40.55,13026.69,Self-Pay,1,0
ADM_000107,813be5a086a1763f,DOC_003,I50.9,2025-02-26,2025-02-28,2025-02-01,Urgent,2,0,7,20,1,100.85,4523.01,Private Commercial,0,0
ADM_000120,82cb95a86d390485,DOC_002,E11.9,2025-02-21,2025-02-27,2025-02-01,Urgent,6,0,23,58,1,104.85,9403.76,Self-Pay,0,1
ADM_000129,3350ccecb6e5a91d,DOC_008,S06.0,2025-11-23,2025-11-27,2025-11-01,Emergency,4,0,22,81,0,92.0,8188.33,Self-Pay,0,0
ADM_000153,acc2176b07698cb7,DOC_008,J44.9,2025-10-24,2025-11-03,2025-10-01,Trauma Center,10,4,21,17,1,60.25,25953.99,Self-Pay,1,1
ADM_000200,4c5ca55a57e2134c,DOC_003,E11.9,2025-12-16,2025-12-20,2025-12-01,Emergency,4,0,18,26,0,104.2,10439.49,Self-Pay,0,0
ADM_000212,14f6c9d295477c74,DOC_004,J44.9,2025-11-08,2025-11-11,2025-11-01,Trauma Center,3,0,25,48,2,74.8,8364.8,Self-Pay,0,1


## 4. Gold star schema and readmission KPIs


In [0]:
gold_fact = (
    silver_fact
    .join(silver_patient, on="patient_hash", how="left")
    .join(silver_doctor, on="doctor_id", how="left")
    .join(silver_diagnosis, on="diagnosis_code", how="left")
)

gold_fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{OUTPUT_SCHEMA}.capstone_gold_fact_admissions"
)
silver_patient.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{OUTPUT_SCHEMA}.capstone_dim_patient"
)
silver_doctor.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{OUTPUT_SCHEMA}.capstone_dim_doctor"
)
silver_diagnosis.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{OUTPUT_SCHEMA}.capstone_dim_diagnosis"
)

readmission_kpis = (
    gold_fact
    .groupBy("admission_month")
    .agg(
        F.count("*").alias("admissions"),
        F.round(F.avg("readmit_30d") * 100, 2).alias("readmission_rate_pct"),
        F.round(F.avg("length_of_stay"), 2).alias("avg_length_of_stay"),
        F.round(F.avg("total_charges"), 2).alias("avg_total_charges"),
    )
    .orderBy("admission_month")
)
readmission_kpis.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{OUTPUT_SCHEMA}.capstone_gold_readmission_kpis"
)
display(readmission_kpis)


admission_month,admissions,readmission_rate_pct,avg_length_of_stay,avg_total_charges
2025-01-01,107,20.56,4.15,10700.0
2025-02-01,87,24.14,3.8,9521.23
2025-03-01,102,24.51,4.34,11069.81
2025-04-01,92,27.17,4.2,10369.42
2025-05-01,97,32.99,3.99,10102.4
2025-06-01,91,28.57,4.33,10583.08
2025-07-01,99,27.27,4.19,10755.99
2025-08-01,105,35.24,4.01,10135.2
2025-09-01,108,28.7,4.24,11180.46
2025-10-01,117,32.48,4.25,10580.54


## 5. Baseline Spark ML readmission model

This is a project demonstration, not a clinically validated decision system.
The held-out recall is measured from the test split; no target is asserted.


In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "length_of_stay",
    "prior_inpatient_admissions",
    "num_medications",
    "num_lab_procedures",
    "num_abnormal_labs",
    "avg_a1c_result",
    "total_charges",
    "icu_stay",
]
model_data = gold_fact.select(
    *[F.coalesce(F.col(name), F.lit(0)).alias(name) for name in feature_columns],
    F.col("readmit_30d").alias("label"),
)

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
classifier = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=50,
    maxDepth=6,
    seed=42,
)
model_pipeline = Pipeline(stages=[assembler, classifier])
train_data, test_data = model_data.randomSplit([0.8, 0.2], seed=42)
if train_data.isEmpty() or test_data.isEmpty():
    raise ValueError("The project dataset must contain enough rows for a train/test split.")

readmission_model = model_pipeline.fit(train_data)
test_predictions = readmission_model.transform(test_data)
positive_recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="recallByLabel",
    metricLabel=1.0,
).evaluate(test_predictions)
print(f"Held-out readmission recall (positive class): {positive_recall:.3f}")
display(test_predictions.select("label", "prediction", "probability").limit(20))


Held-out readmission recall (positive class): 0.101


label,prediction,probability
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7075845148661907"",""0.2924154851338093""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7392087899498696"",""0.26079121005013034""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8488423989392168"",""0.15115760106078324""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8701863479934936"",""0.1298136520065064""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8688478965024906"",""0.13115210349750941""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8551351175662195"",""0.1448648824337804""]}"
1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7960978316663627"",""0.2039021683336372""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6838443101652028"",""0.31615568983479725""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7758927627693366"",""0.22410723723066342""]}"
1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.820143983912674"",""0.17985601608732607""]}"


## 6. Optional live synthetic vitals stream

Change `RUN_LIVE_STREAM` to `True` and run this cell to start a continuous
Spark rate-source demo. It writes synthetic readings to a Delta table and
displays the updating stream. Stop the cell/stream when finished to release
compute. This does not connect to Kafka or use real patient data.


In [0]:
RUN_LIVE_STREAM = False
VITALS_TABLE = f"{OUTPUT_SCHEMA}.capstone_live_vitals"

rate_stream = spark.readStream.format("rate").option("rowsPerSecond", 1).load()
live_vitals = (
    rate_stream
    .withColumn(
        "patient_hash",
        F.sha2(F.concat(F.lit("DEMO_PATIENT_"), (F.col("value") % 20).cast("string")), 256),
    )
    .withColumn("heart_rate", (F.lit(55) + F.pmod(F.col("value") * 7, F.lit(70))).cast("int"))
    .withColumn("spo2", F.round(F.lit(88.0) + F.pmod(F.col("value") * 3, F.lit(13)) * 0.5, 1))
    .withColumn("respiratory_rate", (F.lit(12) + F.pmod(F.col("value") * 5, F.lit(18))).cast("int"))
    .withColumn("systolic_bp", (F.lit(100) + F.pmod(F.col("value") * 11, F.lit(60))).cast("int"))
    .select(
        F.col("timestamp").alias("event_time"),
        "patient_hash",
        "heart_rate",
        "spo2",
        "respiratory_rate",
        "systolic_bp",
    )
)

if RUN_LIVE_STREAM:
    live_query = (
        live_vitals.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{CHECKPOINT_ROOT}/live_vitals")
        .toTable(VITALS_TABLE)
    )
    print(f"Streaming to {VITALS_TABLE}; stop the query when done.")
    display(spark.readStream.table(VITALS_TABLE))
else:
    demo_vitals = (
        spark.range(20)
        .withColumn("event_time", F.current_timestamp())
        .withColumn(
            "patient_hash",
            F.sha2(F.concat(F.lit("DEMO_PATIENT_"), F.col("id").cast("string")), 256),
        )
        .withColumn("heart_rate", (F.lit(55) + F.pmod(F.col("id") * 7, F.lit(70))).cast("int"))
        .withColumn("spo2", F.round(F.lit(88.0) + F.pmod(F.col("id") * 3, F.lit(13)) * 0.5, 1))
        .withColumn("respiratory_rate", (F.lit(12) + F.pmod(F.col("id") * 5, F.lit(18))).cast("int"))
        .withColumn("systolic_bp", (F.lit(100) + F.pmod(F.col("id") * 11, F.lit(60))).cast("int"))
        .drop("id")
    )
    print("Live mode is off. Set RUN_LIVE_STREAM = True to enable the continuous demo.")
    display(demo_vitals)


Live mode is off. Set RUN_LIVE_STREAM = True to enable the continuous demo.


event_time,patient_hash,heart_rate,spo2,respiratory_rate,systolic_bp
2026-10-21T17:20:09.263Z,701365665b04cb9951546a898ae2678e05da3cee6ec6905b722b3b089e8304b4,55,88.0,12,100
2026-10-21T17:20:09.263Z,51acfb7d7ae3e46e8804531df4aa2aba1d8db14947c7de39a86e3e1d7ebf244a,62,89.5,17,111
2026-10-21T17:20:09.263Z,590ca7d63adb9e6295e56143b47451c4270be07028b425c4056a730e83625a54,69,91.0,22,122
2026-10-21T17:20:09.263Z,cb40736e7bd172be2cd21361d4c3d3f04fc69a5d9c32479466eef3566711c714,76,92.5,27,133
2026-10-21T17:20:09.263Z,64257e295995a6eb28af917fcd2c9ca2f453d5bb36167ac6ac61a219d29f1a53,83,94.0,14,144
2026-10-21T17:20:09.263Z,ce3c398ba81b3bbb02eed1bf75699d0f40828d1efc7e24192b05e3a14ea10e0c,90,89.0,19,155
2026-10-21T17:20:09.263Z,292f46df559f83bcb1609e5c7569c541fb5845b717906d66da63c31b14883d03,97,90.5,24,106
2026-10-21T17:20:09.263Z,7afd9a93e8a2b565b4acb157418ca8bb557c553361506b0b3f59c68a2772b1a8,104,92.0,29,117
2026-10-21T17:20:09.263Z,51ce17b01634bdb1e3a466eab54b45512ca2fe9a51fd94334daefc0eea915ad8,111,93.5,16,128
2026-10-21T17:20:09.263Z,3f1fd3e9deb77bf4617987106646eccbc225558d2a6372cd66d46814ff7bee51,118,88.5,21,139


## 7. Dashboard queries

Use the Databricks SQL editor or notebook visualizations to chart the tables:
`workspace.healthcare_capstone.capstone_gold_readmission_kpis` and
`workspace.healthcare_capstone.capstone_live_vitals`. The live table is populated only while
the optional stream is running.
